# DAO Treasury: computational-science simulation

COMSCI/ECON 206 PS2 · Zhou Zimu (FP11). This notebook executes the same standard-library simulation as `python -m src.simulation` and saves three outputs. In a standalone Colab session, its first cell initializes a bundled copy of the exact source and input parameters; in a complete checkout it uses the repository files.

## Model assumptions

One whale (60% stake) and seven ordinary holders (40% total) choose one of three mutually exclusive proposals. Every actor sincerely votes for the proposal with their largest valuation. Valuations are independently uniform on [0, 1), drawn once with seed 206 and reused across mechanisms. Weighted plurality selects the largest score, breaking exact ties A then B then C. SRV transforms each **wallet** stake with its square root; eight Sybil wallets split the whale stake equally and coordinate on the whale’s favorite. Eight economic actors remain eight actors. The browser-based self-play interface is separate from these random profiles.

In [1]:
SOURCE_TEXT = '"""Reproducible, paired Monte Carlo comparison of DAO treasury voting rules.\n\nRun from repository root: python -m src.simulation\nNo third-party packages are required.\n"""\n\nimport csv\nimport json\nimport math\nimport random\nfrom pathlib import Path\n\n\nROOT = Path(__file__).resolve().parents[1]\nPARAMETERS = ROOT / "inputs" / "parameters.json"\nOUTPUT = ROOT / "outputs"\nCONDITIONS = (("TWV_m1", "TWV", 1), ("SRV_m1", "SRV", 1),\n              ("SRV_m8", "SRV", 8))\nMETRICS = ("holder_efficiency", "stake_efficiency", "whale_success",\n           "nonwhale_representation")\n\n\ndef load_parameters(path=PARAMETERS):\n    with open(path, encoding="utf-8") as f:\n        p = json.load(f)\n    assert p["total_actors"] == p["ordinary_holders"] + p["whale_holders"]\n    assert p["whale_holders"] == 1 and p["ordinary_holders"] > 0\n    assert math.isclose(p["whale_token_share"] + p["ordinary_collective_share"], 1)\n    assert p["profiles"] > 0 and p["sybil_wallets"] > 1\n    assert p["proposals"] == ["A", "B", "C"]\n    return p\n\n\ndef profiles(p):\n    """Generate one shared sequence of valuation profiles for all conditions."""\n    rng = random.Random(p["seed"])\n    for number in range(1, p["profiles"] + 1):\n        yield number, [[rng.random() for _ in p["proposals"]]\n                       for _ in range(p["total_actors"])]\n\n\ndef best(values):\n    """Index of maximum; earliest proposal wins exact ties."""\n    return max(range(len(values)), key=lambda i: values[i])\n\n\ndef evaluate(valuations, rule, wallets, p):\n    """Actor 0 is whale. Sybil wallets cast the same sincere vote as their owner."""\n    n = len(p["proposals"])\n    whale_stake = p["whale_token_share"]\n    ordinary_stake = p["ordinary_collective_share"] / p["ordinary_holders"]\n    if rule == "TWV":\n        whale_power = whale_stake\n        ordinary_power = ordinary_stake\n    elif rule == "SRV":\n        whale_power = wallets * math.sqrt(whale_stake / wallets)\n        ordinary_power = math.sqrt(ordinary_stake)\n    else:\n        raise ValueError(rule)\n\n    scores = [0.0] * n\n    votes = [best(v) for v in valuations]\n    scores[votes[0]] += whale_power\n    for vote in votes[1:]:\n        scores[vote] += ordinary_power\n    winner = best(scores)\n\n    holder_welfare = [sum(v[j] for v in valuations) for j in range(n)]\n    stake_welfare = [whale_stake * valuations[0][j] +\n                     ordinary_stake * sum(v[j] for v in valuations[1:])\n                     for j in range(n)]\n    ordinary_welfare = [sum(v[j] for v in valuations[1:]) for j in range(n)]\n    # All denominators are positive with probability one for continuous draws.\n    return {\n        "winner": p["proposals"][winner],\n        "whale_vote": p["proposals"][votes[0]],\n        "winner_score": scores[winner],\n        "whale_power": whale_power,\n        "ordinary_total_power": ordinary_power * p["ordinary_holders"],\n        "holder_efficiency": holder_welfare[winner] / max(holder_welfare),\n        "stake_efficiency": stake_welfare[winner] / max(stake_welfare),\n        "whale_success": int(winner == votes[0]),\n        "nonwhale_representation": int(winner == best(ordinary_welfare)),\n    }\n\n\ndef run(p=None):\n    p = load_parameters() if p is None else p\n    rows = []\n    for number, valuations in profiles(p):\n        for name, rule, wallets in CONDITIONS:\n            rows.append({"profile_id": number, "condition": name,\n                         "rule": rule, "whale_wallets": wallets,\n                         **evaluate(valuations, rule, wallets, p)})\n    comparison = []\n    for name, rule, wallets in CONDITIONS:\n        group = [r for r in rows if r["condition"] == name]\n        comparison.append({"condition": name, "rule": rule,\n                           "whale_wallets": wallets, "profiles": len(group),\n                           **{m: sum(r[m] for r in group) / len(group)\n                              for m in METRICS}})\n    assert len(rows) == p["profiles"] * len(CONDITIONS)\n    assert all(r["whale_success"] == 1 for r in rows if r["condition"] != "SRV_m1")\n    assert all(a["winner"] == b["winner"] for a, b in\n               zip(rows[0::3], rows[2::3]))  # TWV and split SRV coincide here\n    return rows, comparison\n\n\ndef save(rows, comparison, p, directory=OUTPUT):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    with open(directory / "simulation_results.csv", "w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    with open(directory / "mechanism_comparison.csv", "w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=list(comparison[0]))\n        writer.writeheader()\n        writer.writerows(comparison)\n    summary = {\n        "model": "sincere plurality across three mutually exclusive treasury proposals",\n        "parameters": p,\n        "metric_definitions": {\n            "holder_efficiency": "mean over profiles of winning total actor valuation / maximal total actor valuation",\n            "stake_efficiency": "mean over profiles of winning token-share-weighted valuation / maximal token-share-weighted valuation",\n            "whale_success": "fraction of profiles where winner is whale\'s top-valued proposal",\n            "nonwhale_representation": "fraction where winner maximizes seven ordinary holders\' combined valuation",\n        },\n        "results": {row["condition"]: {m: row[m] for m in METRICS}\n                    for row in comparison},\n        "observation": "SRV with eight equal whale wallets yields whale voting power above all ordinary holders combined; TWV and split SRV choose the whale\'s sincere first choice on every paired profile.",\n    }\n    with open(directory / "summary.json", "w", encoding="utf-8") as f:\n        json.dump(summary, f, indent=2, ensure_ascii=False)\n        f.write("\\n")\n    return summary\n\n\ndef main():\n    p = load_parameters()\n    rows, comparison = run(p)\n    save(rows, comparison, p)\n    print("condition,holder_efficiency,stake_efficiency,whale_success,nonwhale_representation")\n    for row in comparison:\n        print(",".join([row["condition"]] + [f"{row[m]:.6f}" for m in METRICS]))\n    print(f"Saved {len(rows)} condition-profile rows to {OUTPUT}")\n\n\nif __name__ == "__main__":\n    main()\n'
PARAMETER_TEXT = '{\n  "seed": 206,\n  "profiles": 1000,\n  "total_actors": 8,\n  "ordinary_holders": 7,\n  "whale_holders": 1,\n  "whale_token_share": 0.60,\n  "ordinary_collective_share": 0.40,\n  "proposals": ["A", "B", "C"],\n  "sybil_wallets": 8,\n  "valuation_distribution": "independent uniform [0, 1) per actor and proposal",\n  "tie_break": "proposal order A, B, C"\n}\n'
from pathlib import Path
root = Path.cwd()
if not (root / 'src' / 'simulation.py').exists():
    (root / 'src').mkdir(exist_ok=True)
    (root / 'inputs').mkdir(exist_ok=True)
    (root / 'src' / '__init__.py').write_text('', encoding='utf-8')
    (root / 'src' / 'simulation.py').write_text(SOURCE_TEXT, encoding='utf-8')
    (root / 'inputs' / 'parameters.json').write_text(PARAMETER_TEXT, encoding='utf-8')
    print('Initialized bundled model for standalone notebook run.')
from pathlib import Path
import csv
import json
from src.simulation import CONDITIONS, METRICS, load_parameters, run, save

root = Path.cwd()
assert (root / 'src' / 'simulation.py').exists()
p = load_parameters()
print(json.dumps(p, indent=2))

{
  "seed": 206,
  "profiles": 1000,
  "total_actors": 8,
  "ordinary_holders": 7,
  "whale_holders": 1,
  "whale_token_share": 0.6,
  "ordinary_collective_share": 0.4,
  "proposals": [
    "A",
    "B",
    "C"
  ],
  "sybil_wallets": 8,
  "valuation_distribution": "independent uniform [0, 1) per actor and proposal",
  "tie_break": "proposal order A, B, C"
}


## Simulation procedure

For each of 1,000 paired profiles, draw eight-by-three valuations, compute each actor’s top proposal, then score TWV, one-wallet SRV, and eight-wallet SRV. The whale’s Sybil identities retain the same aggregate stake and preference. The comparison averages each per-profile metric.

In [2]:
rows, comparison = run(p)
assert len(rows) == 3 * p['profiles']
assert [c['condition'] for c in comparison] == [c[0] for c in CONDITIONS]
assert all(rows[i]['profile_id'] == rows[i+1]['profile_id'] == rows[i+2]['profile_id'] for i in range(0, len(rows), 3))
assert all(rows[i]['winner'] == rows[i+2]['winner'] for i in range(0, len(rows), 3))
print(f"{p['profiles']} paired profiles; {len(rows)} condition-profile rows")
print('TWV and eight-wallet SRV winners agree in all 1,000 profiles')

1000 paired profiles; 3000 condition-profile rows
TWV and eight-wallet SRV winners agree in all 1,000 profiles


## Metric definitions and mechanism comparison

Holder efficiency = winning total actor valuation divided by best total actor valuation, averaged across profiles. Stake efficiency applies actual token shares to actor valuations before taking that ratio. Whale success is the fraction where the whale’s favorite wins. Non-whale representation is the fraction where the winner maximizes the seven ordinary actors’ **combined** valuation; it does not count seven separate wins.

In [3]:
print('condition       holder_eff  stake_eff   whale_success  nonwhale_rep')
for c in comparison:
    print(f"{c['condition']:<15} {c['holder_efficiency']:.6f}     {c['stake_efficiency']:.6f}     {c['whale_success']:.3f}          {c['nonwhale_representation']:.3f}")
print('SRV one wallet minus TWV, non-whale representation:', round(comparison[1]['nonwhale_representation'] - comparison[0]['nonwhale_representation'], 3))

condition       holder_eff  stake_eff   whale_success  nonwhale_rep
TWV_m1          0.899065     0.991097     1.000          0.321
SRV_m1          0.927781     0.977076     0.896          0.417
SRV_m8          0.899065     0.991097     1.000          0.321
SRV one wallet minus TWV, non-whale representation: 0.096


## Save and inspect outputs

The detailed CSV includes each outcome and per-profile metric. Valuations can be regenerated from the committed seed and generator; they are not duplicated in the CSV. The comparison CSV and summary JSON are calculated from those same rows.

In [4]:
summary = save(rows, comparison, p)
with open(root / 'outputs' / 'simulation_results.csv', newline='', encoding='utf-8') as f:
    detailed = list(csv.DictReader(f))
with open(root / 'outputs' / 'mechanism_comparison.csv', newline='', encoding='utf-8') as f:
    stored = list(csv.DictReader(f))
with open(root / 'outputs' / 'summary.json', encoding='utf-8') as f:
    disk_summary = json.load(f)
assert len(detailed) == 3000 and len(stored) == 3
assert disk_summary == summary
for observed in stored:
    for metric in METRICS:
        assert abs(float(observed[metric]) - summary['results'][observed['condition']][metric]) < 1e-12
print('Saved and re-read 3,000 detailed rows, 3 comparison rows, and a matching summary JSON.')
print('First profile winners:', [(r['condition'], r['winner']) for r in detailed[:3]])

Saved and re-read 3,000 detailed rows, 3 comparison rows, and a matching summary JSON.
First profile winners: [('TWV_m1', 'C'), ('SRV_m1', 'C'), ('SRV_m8', 'C')]


## Interpretation

With one wallet, SRV increases non-whale representation from 0.321 to 0.417 and holder efficiency from 0.899065 to 0.927781 in this sample. Stake efficiency declines from 0.991097 to 0.977076. Splitting 0.60 into eight equal SRV wallets gives the whale total weight √(8 × 0.60) ≈ 2.190890, which exceeds all seven ordinary holders’ combined weight ≈ 1.673320. The whale therefore determines all outcomes in that condition, just as in TWV here. This is a paired synthetic comparison, not evidence from participants or a claim of strategic equilibrium.

## Limitations

The result is conditional on one seed and 1,000 profiles, uniform independent valuations, sincere votes, equal ordinary stakes, three alternatives, and zero-cost coordinated Sybil identities. Different distributions, voting behavior, identity checks, abstentions, or treasury outcomes can change the magnitudes. The page’s older numbers lacked source code or a specified stochastic model and are reconciled in `VERIFICATION.md`.